# Experiments

This file contains multiple experiments that were done in order to find the solution to the [task](task.md). Most of the necesary code is located in [solution.py](solution.py).

## Preparation

First we load and prepare the data. Preparation includes:
- removing whitespaces
- assigning labels for cheap, average and expensive houses
- converting categorical data to numerical values

In [10]:
from solution import prepare_train_data, NetConfig, NeuralNetwork
import numpy as np

X, y = prepare_train_data("train_data.csv")
X, y = np.array(X), np.array(y)

Below are the methods we will use to compare the quality of predictions of created models.
- `calc_accuracy` measures average accuracy across all price categories (in our case: 0, 1, 2)
- `evaluate_cv` estimates model performance using 5-fold stratified cross-validation

In [11]:
from sklearn.model_selection import StratifiedKFold
import copy

def calc_accuracy(preds, targets) -> float:
    return np.mean([
        (preds[targets == i] == i).mean()
        for i in range(3)
        if (targets == i).sum() > 0
    ])


def evaluate_cv(name: str, model, X, y) -> None:
    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    for tr, te in kf.split(X, y):
        m = copy.deepcopy(model)
        m.fit(X[tr], y[tr])
        scores.append(calc_accuracy(m.predict(X[te]), y[te]))
    
    print(f"{name}: {np.mean(scores):.4f} ± {np.std(scores):.4f}")

## Experiments

### 1. Managing unbalanced dataset
Since the task specifies that the dataset favours certain house classes and is unbalanced, we need to find a way to make our model prepared for that.

In [15]:
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTETomek

model1 = NeuralNetwork(NetConfig(sampler=None))
evaluate_cv("No balancing", model1, X, y)

model2 = NeuralNetwork(NetConfig(sampler=SMOTE))
evaluate_cv("SMOTE", model2, X, y)

model3 = NeuralNetwork(NetConfig(sampler=RandomUnderSampler))
evaluate_cv("RandomUnderSampler", model3, X, y)

model5 = NeuralNetwork(NetConfig(sampler=None, class_weight=True))
evaluate_cv("Class weights", model5, X, y)


No balancing: 0.7479 ± 0.0157
SMOTE: 0.8734 ± 0.0060
RandomUnderSampler: 0.8715 ± 0.0061
Class weights: 0.8802 ± 0.0033


We can see that **balancing the dataset does improve the results**. 
- Without any balancing the model scores  **0.749**, while every balancing method jumps to around **0.87–0.88**. 
- The best result comes from **Class Weight (0.8826 ± 0.0043)**, which we will use as the default for all further experiments.

### 2. Network layers and their activation

In [14]:
model1 = NeuralNetwork(NetConfig(class_weight=True, layers=[256, 128, 64], activation=[nn.ReLU]))
evaluate_cv("[256,128,64] with ReLU", model3a, X, y)

model2 = NeuralNetwork(NetConfig(class_weight=True, layers=[256, 128, 64], activation=[nn.Identity]))
evaluate_cv("[256,128,64] with linear activation", model2, X, y)

model3 = NeuralNetwork(NetConfig(class_weight=True, layers=[256, 128, 64], activation=[nn.LeakyReLU]))
evaluate_cv("[256,128,64] with LeakyReLU", model3, X, y)

model4 = NeuralNetwork(NetConfig(class_weight=True, layers=[256, 128, 64], activation=[nn.Tanh]))
evaluate_cv("[256,128,64] with Tanh", model4, X, y)

model5 = NeuralNetwork(NetConfig(class_weight=True, layers=[256, 128, 64], activation=[nn.ReLU, nn.ReLU, nn.LeakyReLU]))
evaluate_cv("[256,128,64] with ReLU/LeakyReLU", model5, X, y)


[256,128,64] with ReLU: 0.8854 ± 0.0049
[256,128,64] with linear activation: 0.8798 ± 0.0046
[256,128,64] with LeakyReLU: 0.8862 ± 0.0070
[256,128,64] with Tanh: 0.8859 ± 0.0048
[256,128,64] with ReLU/LeakyReLU: 0.8847 ± 0.0053


All three activation configurations perform nearly identically (around 0.885), so the choice of activation function has no meaningful impact here. We will stick with the default **ReLU** for simplicity.

### 3. Adding Batch Normalization
masz o tym info w tych materiałach do lab4

### 4. Different Dropouts
i o tym też. 

Jeśli te eksperymenty z dropout i BN wyjda takie ze wszystko takie same wyniki daje i nie ma jakiejs poprawy to mozna je wywalic. 

### 5. Grid Search
Wypadało by puścić jakiegoś grid searcha żeby zmierzyć, który NetConfig daje najlepszy model. Albo jakąs mądrzejsza niż Grid search metodę wymyślić.

## Conclusion
Tak czy siak trzeba znaleźć najlepsze kombo parametrów i potem wykorzystać je w  mainie solution.py żeby wygenerować te preds.csv. Fajnie było by kiedykolwiek przekroczyć accuracy 0.9 ale nwm jak to osiągnąć tbh